# Chapter 13 — Recursive Algorithms for Speed

**Source:** Wengrow, *A Common-Sense Guide to Data Structures and Algorithms*, Ch. 13

## The big idea
Chapter 12 used recursion/DP to *fix* slow code. This chapter uses recursion to build algorithms that are **fundamentally faster** than the simple sorts (Bubble, Selection, Insertion are all O(N²) on average).

| Algorithm | What it does | Average case |
|---|---|---|
| **Quicksort** | sorts an array | **O(N log N)** |
| **Quickselect** | finds the k-th lowest value *without* sorting everything | **O(N)** |

Both are built on one building block: **partitioning**.

## 1. Partitioning

Pick a **pivot** (we always use the **rightmost** element). After a partition:
- everything **smaller** than the pivot is to its **left**
- everything **bigger** is to its **right**
- the pivot is now in its **final, correct position** (the rest are not sorted yet!)

### The algorithm (two pointers)
1. **left pointer** starts at the first cell; **right pointer** starts at the cell just left of the pivot.
2. Move `left` → right until it hits a value **≥ pivot**, then stop.
3. Move `right` → left until it hits a value **≤ pivot**, then stop.
4. If `left` has reached or passed `right` → go to step 5. Otherwise **swap** the two values, move `left` one step, repeat from 2.
5. **Swap the pivot with the value at `left`.** Return `left` (the pivot's final index).

### Cost of one partition: O(N)
- Comparisons: ~N (every element is compared to the pivot).
- Swaps: at most N/2, about N/4 on random data.
- Total ≈ 1.25N steps → **O(N)**.

**Why does a partition take an index range, not the whole array?**
Quicksort will call it on *sub-sections* of the array, so `left` and `right` must be parameters.

In [ ]:
def partition(arr, left, right):
    """Partition arr[left..right] in place around the rightmost value.
    Returns the final index of the pivot."""
    pivot_index = right
    pivot = arr[pivot_index]
    right -= 1                                   # right pointer starts just left of pivot

    while True:
        while arr[left] < pivot:                 # move left pointer until value >= pivot
            left += 1
        while right > left and arr[right] > pivot:   # move right pointer until value <= pivot
            right -= 1                           # (guard: see note below)

        if left >= right:                        # pointers met or crossed -> done moving
            break
        arr[left], arr[right] = arr[right], arr[left]   # swap
        left += 1

    arr[left], arr[pivot_index] = arr[pivot_index], arr[left]   # pivot to its final place
    return left


a = [0, 5, 2, 1, 6, 3]
idx = partition(a, 0, len(a) - 1)
print(idx, a)    # pivot 3 ends at index 3: smaller values left, bigger right

## 2. Quicksort

Partitioning + recursion:

1. **Partition** the array → the pivot is now in its final place.
2. Treat the part **left of the pivot** and the part **right of the pivot** as their own arrays. **Recursively** quicksort each.
3. **Base case:** a sub-array with **0 or 1 elements** is already sorted → do nothing.

Notice there is no "combine" step. Each partition places one more element permanently, so when the recursion bottoms out, the whole array is sorted.

Link to earlier chapters: the left/right-of-pivot calls are the same "call it on a smaller piece" pattern from Chapter 10. The difference is that here the pieces don't overlap, so no memoization is needed.

In [ ]:
def quicksort(arr, left=0, right=None):
    if right is None:
        right = len(arr) - 1

    if right - left <= 0:                         # base case: 0 or 1 elements
        return

    pivot_index = partition(arr, left, right)     # step 1
    quicksort(arr, left, pivot_index - 1)         # step 2: left of pivot
    quicksort(arr, pivot_index + 1, right)        # step 2: right of pivot


a = [0, 5, 2, 1, 6, 3]
quicksort(a)
print(a)    # [0, 1, 2, 3, 5, 6]

## 3. Efficiency of Quicksort

**Best / average case: O(N log N)**
- Every partition costs about N steps in total across one "level" of sub-arrays.
- If the pivot lands near the middle, the array is halved each time → it takes **log N** levels to get down to size 1.
- So: **N steps per level × log N levels = N log N**.

| N | 8 | 16 | 32 | 64 |
|---|---|---|---|---|
| log N | 3 | 4 | 5 | 6 |
| N · log N | 24 | 64 | 160 | 384 |

(The book's table says ~24, ~64, ~160 steps for N = 8, 16, 32. Same pattern: steps ≈ N · log N.)

**Worst case: O(N²)**
- Happens when the pivot always ends up at one **end**, which is what you get on an **already sorted** or **reverse-sorted** array with this "rightmost pivot" rule.
- Partition sizes: N + (N−1) + (N−2) + … + 1 ≈ N²/2 → O(N²).

Surprising bit: for Quicksort, a sorted array is the **worst** case. Well-mixed data is the best.

| | Best | Average | Worst |
|---|---|---|---|
| Insertion Sort | O(N) | O(N²) | O(N²) |
| **Quicksort** | O(N log N) | **O(N log N)** | O(N²) |

Insertion Sort wins on already-sorted data, but **average** is what happens most of the time. That's why Quicksort is used under the hood of many built-in sorts.

## 4. Quickselect

**Problem:** find the k-th lowest value (the median, the 25th percentile, …) in an unsorted array. You don't need the whole array sorted.

**Naive way:** sort, then index → O(N log N).
**Better:** Quickselect, a hybrid of **Quicksort** (partitioning) and **binary search** (throw half away).

How it works:
1. Partition the array → the pivot lands at its final index `p`.
2. Compare `p` with the index `k` you want:
   - `k == p` → the pivot **is** the answer.
   - `k < p` → the answer is in the **left** part. Recurse only on the left.
   - `k > p` → the answer is in the **right** part. Recurse only on the right.
3. Base case: a sub-array with one element is the answer.

Why O(N) and not O(N log N): Quicksort recurses into **both** halves, Quickselect into **one**.
The work is N + N/2 + N/4 + … ≈ **2N → O(N)** on average.

**Index convention:** `k` is a 0-based index. `k=0` is the lowest, `k=1` the second-lowest, and so on.

In [ ]:
def quickselect(arr, k, left=0, right=None):
    """Return the k-th lowest value (0-based) of arr. Rearranges arr in place."""
    if right is None:
        right = len(arr) - 1

    if right - left <= 0:                  # one cell left: that's our answer
        return arr[left]

    pivot_index = partition(arr, left, right)

    if k < pivot_index:                    # answer is left of pivot
        return quickselect(arr, k, left, pivot_index - 1)
    elif k > pivot_index:                  # answer is right of pivot
        return quickselect(arr, k, pivot_index + 1, right)
    else:                                  # k == pivot_index
        return arr[pivot_index]


nums = [0, 50, 20, 10, 60, 30]
print(quickselect(nums, 1))     # second-lowest -> 10

## 5. Sorting as a key to other algorithms

The fastest known general sorts are **O(N log N)** (Quicksort, Mergesort). So:

> If you use sorting as one step of a bigger algorithm, that algorithm is **at least O(N log N)**, and that is often a big win over O(N²).

**Example: has-duplicates**

| Approach | Time | Extra memory |
|---|---|---|
| Nested loops | O(N²) | none |
| Hash set (Ch. 8) | O(N) | O(N) |
| **Sort first, then compare neighbours** | **O(N log N)** | little |

Sorting puts equal values next to each other, so one pass checking `arr[i] == arr[i+1]` finds any duplicate.
Cost: `N log N` (sort) `+ N` (loop) → keep the larger term → **O(N log N)**.

(The book mentions Mergesort as another O(N log N) sort and recommends looking it up. It's worth doing later.)

In [ ]:
def has_duplicate(arr):
    arr = sorted(arr)                       # O(N log N) -- sorted() returns a copy
    for i in range(len(arr) - 1):           # O(N)
        if arr[i] == arr[i + 1]:
            return True
    return False

print(has_duplicate([5, 9, 3, 2, 4, 5, 6]))   # True
print(has_duplicate([1, 2, 3]))               # False

## Summary

- **Partition**: pick a pivot, put smaller left, bigger right → the pivot is in its final place. **O(N)**.
- **Quicksort** = partition + recurse on both sides. **O(N log N)** average, **O(N²)** worst (already-sorted input with a rightmost pivot).
- **Quickselect** = partition + recurse on **one** side only. **O(N)** average.
- **Sort first** is a general trick: it can turn an O(N²) algorithm into O(N log N).
- In real code you'd use `sorted()` / `list.sort()` (Timsort in Python), not hand-written Quicksort. The value here is the *technique*: recursion + halving.

## Exercises (book solutions: p. 451)

### Exercise 1: greatest product of three numbers
**Input:** a list of **positive** numbers, e.g. `[5, 2, 9, 1, 7]`.
**Output:** the biggest product you can get by multiplying **any three** of them (here: 5 × 7 × 9 = 315).
**Naive:** three nested loops → O(N³).
**Your job:** use **sorting** to get O(N log N).

### Exercise 2: missing number
**Input:** a list that should contain every integer from `0` up to the list's length, but **one is missing**. Examples: `[5, 2, 4, 1, 0]` → missing `3`; `[9, 3, 2, 5, 6, 7, 1, 0, 4]` → missing `8`.
**Output:** the missing number.
**Naive version below:** `i in array` is itself O(N) and it runs inside a loop → O(N²).
**Your job:** use **sorting** to get O(N log N).

### Exercise 3: greatest number, three ways
**Input:** a list of numbers, e.g. `[4, 9, 2, 7]`.
**Output:** the largest one, written as **three separate functions**:
1. one that is **O(N²)**
2. one that is **O(N log N)**
3. one that is **O(N)**
(Think: what would you do with nested loops? With sorting? With a single pass?)

In [ ]:
# Exercise 1 — implement with sorting, O(N log N)
def greatest_product_of_three(array):
    pass

print(greatest_product_of_three([5, 2, 9, 1, 7]))          # expect 315
print(greatest_product_of_three([3, 3, 3]))                # expect 27

In [ ]:
# Exercise 2 — naive O(N^2) version given by the book, translated to Python
def find_missing_number_slow(array):
    for i in range(len(array)):
        if i not in array:                 # `in` on a list is O(N)
            return i
    return None

print(find_missing_number_slow([5, 2, 4, 1, 0]))                 # 3
print(find_missing_number_slow([9, 3, 2, 5, 6, 7, 1, 0, 4]))     # 8

# Now write the O(N log N) version using sorting:
def find_missing_number(array):
    pass

In [ ]:
# Exercise 3 — three different implementations
def greatest_n_squared(array):
    pass

def greatest_n_log_n(array):
    pass

def greatest_n(array):
    pass

for f in (greatest_n_squared, greatest_n_log_n, greatest_n):
    print(f.__name__, f([4, 9, 2, 7]))     # expect 9 each time